<a href="https://colab.research.google.com/github/nihedzaoui/flyrank-ml-internship/blob/main/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nihedzaoui/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

This case study addresses a practical FlyRank content-review problem: when a team has more existing pages to review than it can inspect manually, which pages should be prioritized for refresh review?

**Research question:** Can a leakage-safe machine-learning ranking concentrate observed declining pages near the top of the review queue more effectively than a transparent rule-based baseline?

The decision supported by the model is **which pages to inspect first**. The model does not automatically decide which pages should be refreshed.

In [ ]:
# Define the decision supported by the study.
decision = "Prioritize existing content pages for human refresh review"

print("Decision supported:")
print(decision)

Decision supported:
Prioritize existing content pages for human refresh review


## 2. Data

The analysis uses the reproducible public-safe release containing **30,000 pseudonymized content items and 44 columns**.

The release contains search-performance, engagement, freshness, and content-related signals.

The declining label is defined as:

`is_declining_label = (trend_direction == "down")`

The observed declining-label rate is **54.2%**.

To avoid leakage, `trend_direction` and `trend_pct` are excluded because they contain outcome-derived information. Identifiers such as `content_id` and `client_id` are also excluded from the model features.

The public artifact does not expose private production rows, client names, URLs, page titles, keywords, or raw queries.

In [ ]:
!git clone https://github.com/nihedzaoui/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 218, done.
remote: Counting objects: 100% (218/218), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 218 (delta 93), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (218/218), 2.03 MiB | 9.54 MiB/s, done.
Resolving deltas: 100% (93/93), done.


In [ ]:
from pathlib import Path

ROOT = Path("/content/flyrank-ml-internship")
DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"

print("Repository exists:", ROOT.exists())
print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)

Repository exists: True
Dataset exists: True
Dataset path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

# Locate the repository
ROOT = Path("/content/flyrank-ml-internship")

if not ROOT.exists():
    ROOT = Path.cwd()

DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"

# Load the public-safe anonymized release
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

# Basic structure
print("\nFirst 5 rows:")
display(df.head())

# Target definition
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("\nTarget distribution:")
target_summary = pd.DataFrame({
    "count": df["is_declining_label"].value_counts().sort_index(),
    "share": df["is_declining_label"].value_counts(normalize=True).sort_index()
})

target_summary.index = ["not_declining", "declining"]
display(target_summary)

print(
    f"\nObserved declining-label rate: "
    f"{df['is_declining_label'].mean():.3f}"
)

# Required columns used by the pipeline
required_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction",
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

print("\nRequired-column check:")
if missing_columns:
    print("Missing:", missing_columns)
else:
    print("All required columns are present.")

# Leakage / identifier fields excluded from modeling
excluded_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
]

print("\nExcluded from model features:")
for column in excluded_columns:
    print(f"- {column}")

# Public-safety check
print("\nPublic-safety checks:")
print("Client names/domains/URLs/page titles/raw queries are not used as model features.")

Dataset shape: (30000, 44)
Number of rows: 30000
Number of columns: 44

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Target distribution:


,count,share
not_declining,13738,0.457933
declining,16262,0.542067



Observed declining-label rate: 0.542

Required-column check:
All required columns are present.

Excluded from model features:
- content_id
- client_id
- trend_direction
- trend_pct
- is_declining_label

Public-safety checks:
Client names/domains/URLs/page titles/raw queries are not used as model features.


## 3. Methodology

The analysis treats content refresh as a ranking problem. A transparent rule-based baseline is compared with machine-learning models, with the Random Forest selected using Precision@50.

The target is `is_declining_label`, defined from the observed `trend_direction` field. Target-derived trend fields and identifiers are excluded from the model features.

The model pipeline uses numerical and categorical content-performance signals. Validation uses a client-holdout design when possible, keeping clients separated between training and testing.

The final ranking score is based on the positive-class probability produced by the selected model.

In [ ]:

import subprocess
import sys

# Run the same feature-preparation and baseline/modeling pipeline
# used by the repository.

scripts_dir = ROOT / "scripts"

def run_script(script_name):
    print(f"\n{'=' * 60}")
    print(f"Running {script_name}")
    print(f"{'=' * 60}")

    subprocess.run(
        [sys.executable, str(scripts_dir / script_name)],
        cwd=ROOT,
        check=True
    )

# 1. Prepare leakage-safe features
run_script("01_prepare_features.py")

# 2. Build the transparent baseline
run_script("02_baseline_score.py")

# 3. Train and compare the models
run_script("03_train_model.py")

print("\nPipeline steps 1–3 completed successfully.")



Running 01_prepare_features.py

Running 02_baseline_score.py

Running 03_train_model.py

Pipeline steps 1–3 completed successfully.


## 4. Results (vs baseline)

The model and the transparent rule baseline are evaluated on the same held-out test set.

Precision@50 is the main ranking metric because the operational decision is to identify which pages should be inspected first when review capacity is limited.

The comparison below reports the measured results produced by the repository pipeline.

In [ ]:

import json
import pandas as pd

RESULTS_PATH = ROOT / "outputs" / "model_results.json"

with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

print("Validation information")
print("----------------------")
print("Split strategy:", results["split_strategy"])
print("Training rows:", results["train_rows"])
print("Test rows:", results["test_rows"])
print("Feature count:", results["feature_count"])

# Build the comparison table
rows = []

for model_name, metrics in results["models"].items():
    rows.append({
        "Model": model_name,
        "ROC-AUC": metrics["roc_auc"],
        "Average Precision": metrics["average_precision"],
        "Precision@50": metrics["precision_at_50"],
        "Recall": metrics["recall"],
        "F1": metrics["f1"],
    })

baseline = results["baseline"]

rows.append({
    "Model": "baseline_rules",
    "ROC-AUC": baseline["baseline_roc_auc"],
    "Average Precision": baseline["baseline_average_precision"],
    "Precision@50": baseline["baseline_precision_at_50"],
    "Recall": np.nan,
    "F1": np.nan,
})

results_table = pd.DataFrame(rows)

print("\nModel comparison:")
display(
    results_table.style.format({
        "ROC-AUC": "{:.3f}",
        "Average Precision": "{:.3f}",
        "Precision@50": "{:.3f}",
        "Recall": "{:.3f}",
        "F1": "{:.3f}",
    })
)

# Identify the selected model
print("\nSelected model:")
print(results["best_model"]["name"])

print(
    "\nSelection metric:",
    results["best_model"]["selection_metric"]
)

# Top feature importance
importance = pd.DataFrame(
    results["best_model"]["feature_importance_top"]
)

print("\nTop model features:")
display(importance.head(10))


Validation information
----------------------
Split strategy: client_holdout
Training rows: 27675
Test rows: 2325
Feature count: 52

Model comparison:


,Model,ROC-AUC,Average Precision,Precision@50,Recall,F1
0,decision_tree,0.742,0.575,0.580,0.716,0.634
1,logistic_regression,0.700,0.522,0.400,0.567,0.566
2,random_forest,0.750,0.618,0.740,0.744,0.640
3,baseline_rules,0.627,0.468,0.240,nan,nan



Selected model:
random_forest

Selection metric: precision_at_50

Top model features:


,feature,importance
0,days_with_impressions,0.158144
1,log_impressions_90d,0.128638
2,avg_position,0.109164
3,content_age_days,0.095168
4,char_count,0.042608
5,word_count,0.039609
6,log_clicks_90d,0.034463
7,ctr,0.033295
8,scroll_rate,0.031226
9,days_with_sessions,0.027995


## 5. Limitations

The analysis cannot establish that refreshing content causes future search-performance improvement.

The target is a proxy derived from observed trend information rather than a direct measure of editorial quality. The dataset is observational, and the public artifact represents the reproducible anonymized release rather than private production data.

The model output is a ranking signal for human review, not a calibrated business-outcome probability or an autonomous publishing decision.

In [ ]:

# Programmatic limitation and safety checks

feature_path = ROOT / "data" / "processed" / "refresh_feature_vector.csv"

if feature_path.exists():
    feature_df = pd.read_csv(feature_path)

    forbidden_features = {
        "content_id",
        "client_id",
        "trend_direction",
        "trend_pct",
        "is_declining_label",
    }

    feature_columns = set(
        results["model_numeric_features"]
        + results["model_categorical_features"]
    )

    leakage_fields_present = sorted(
        forbidden_features.intersection(feature_columns)
    )

    print("Leakage audit")
    print("------------")
    print("Number of model features:", len(feature_columns))

    if leakage_fields_present:
        print("WARNING — forbidden fields detected:")
        print(leakage_fields_present)
    else:
        print("PASS — identifiers and target-derived fields are excluded.")

    print("\nValidation:")
    print("Split strategy:", results["split_strategy"])
    print("Training rows:", results["train_rows"])
    print("Test rows:", results["test_rows"])

    print("\nInterpretation:")
    print("- The target is an observed-trend proxy.")
    print("- The data is observational.")
    print("- The output is decision support for human review.")
else:
    print("Processed feature file not found.")


Leakage audit
------------
Number of model features: 26
PASS — identifiers and target-derived fields are excluded.

Validation:
Split strategy: client_holdout
Training rows: 27675
Test rows: 2325

Interpretation:
- The target is an observed-trend proxy.
- The data is observational.
- The output is decision support for human review.


## 6. Ranked recommendations

The ranked queue is intended to help reviewers decide which existing pages to inspect first.

The recommendations are ordered by the final refresh score and supported by observable signals and reason codes. They remain subject to human verification.

1. Review high-confidence candidates first.
2. Verify impressions, CTR, engagement, position, freshness, and editorial context.
3. Separate CTR and engagement diagnoses before selecting an intervention.
4. Keep a human approval gate before any content change.
5. Monitor outcomes prospectively and use time-aware or experimental designs before making causal claims.

In [ ]:
from pathlib import Path

ROOT = Path("/content/flyrank-ml-internship")

print("Repository:", ROOT)
print("\nOutputs directory:")

outputs_dir = ROOT / "outputs"

if outputs_dir.exists():
    for path in sorted(outputs_dir.rglob("*")):
        if path.is_file():
            print(path.relative_to(ROOT))
else:
    print("outputs/ directory does not exist.")

Repository: /content/flyrank-ml-internship

Outputs directory:
outputs/charts/action_mix.svg
outputs/charts/confidence_mix.svg
outputs/charts/top_feature_importance.svg
outputs/charts/top_reason_codes.svg
outputs/charts/trend_distribution.svg
outputs/model_report.md
outputs/model_results.json
outputs/refresh_queue_sample.csv


In [ ]:
print("\nCSV files in repository:")

for path in sorted(ROOT.rglob("*.csv")):
    print(path.relative_to(ROOT))


CSV files in repository:
data/processed/baseline_refresh_queue.csv
data/processed/model_predictions.csv
data/processed/refresh_feature_vector.csv
data/raw/content_refresh_anonymized.csv
outputs/refresh_queue_sample.csv


In [ ]:
import subprocess

result = subprocess.run(
    ["grep", "-R", "refresh_queue", "-n", str(ROOT)],
    capture_output=True,
    text=True
)

print(result.stdout[:10000])

/content/flyrank-ml-internship/.gitignore:19:# refresh_queue_sample.csv), ignore regenerated heavy artifacts
/content/flyrank-ml-internship/.gitignore:24:!outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship/scripts/04_evaluate_and_export.py:21:BASELINE_PATH = PROCESSED_DIR / "baseline_refresh_queue.csv"
/content/flyrank-ml-internship/scripts/04_evaluate_and_export.py:24:QUEUE_PATH = OUTPUT_DIR / "refresh_queue.csv"
/content/flyrank-ml-internship/scripts/04_evaluate_and_export.py:265:- `outputs/refresh_queue.csv`
/content/flyrank-ml-internship/scripts/03_train_model.py:35:BASELINE_PATH = PROCESSED_DIR / "baseline_refresh_queue.csv"
/content/flyrank-ml-internship/scripts/02_baseline_score.py:13:OUTPUT_PATH = PROCESSED_DIR / "baseline_refresh_queue.csv"
/content/flyrank-ml-internship/scripts/05_build_pdf_report.py:27:QUEUE_PATH = OUTPUT_DIR / "refresh_queue.csv"
/content/flyrank-ml-internship/scripts/05_build_pdf_report.py:625:                ["1", "Manually inspect the top 25

In [ ]:
for path in ROOT.rglob("*"):
    if path.is_file() and path.suffix in [".py", ".ipynb", ".md", ".txt"]:
        try:
            text = path.read_text(errors="ignore")
            if "refresh_queue" in text:
                print("\n---", path.relative_to(ROOT), "---")
                for i, line in enumerate(text.splitlines(), 1):
                    if "refresh_queue" in line:
                        print(f"{i}: {line}")
        except Exception:
            pass


--- GUIDE.md ---
21: | `outputs/` | Pipeline results. Three **committed examples** show the target shape (`model_report.md`, `refresh_queue_sample.csv`, `charts/`); everything else regenerates | Regenerated files are gitignored — that's intentional (see FAQ) |
42: data/processed/baseline_refresh_queue.csv
50: outputs/refresh_queue.csv, outputs/model_report.md, outputs/charts/
69: `outputs/refresh_queue.csv`, the PDF). Only three example outputs are committed so you can see

--- w05_model.ipynb ---
98:             "/content/flyrank-ml-internship/outputs/refresh_queue_sample.csv\n",

--- 01_first_look_and_discovery.ipynb ---
124:             "Wrote baseline queue: /content/flyrank-ml-internship-starter/flyrank-ml-internship/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv\n",
139:             "Wrote final refresh queue: /content/flyrank-ml-internship-starter/flyrank-ml-internship/flyrank-ml-internship/outputs/refresh_queue.csv\n",
151:             "Queue: outputs/refresh_

## 7. Artifacts the paper embeds

The paper reuses aggregate artifacts generated by the modeling and evaluation pipeline:

- model comparison metrics;
- feature-importance information;
- refresh queue confidence distribution;
- suggested-action distribution;
- reason-code distribution;
- trend distribution.

These artifacts are generated from the public-safe anonymized release and contain no client names, URLs, page titles, keywords, or raw search queries.

In [ ]:

from pathlib import Path

# Generate the final queue, report, and charts from the trained outputs.
run_script("04_evaluate_and_export.py")

CHART_DIR = ROOT / "outputs" / "charts"

expected_artifacts = [
    "action_mix.svg",
    "confidence_mix.svg",
    "top_reason_codes.svg",
    "top_feature_importance.svg",
    "trend_distribution.svg",
]

print("\nPaper artifacts")
print("---------------")

artifact_status = []

for filename in expected_artifacts:
    path = CHART_DIR / filename

    artifact_status.append({
        "artifact": filename,
        "exists": path.exists(),
        "path": str(path.relative_to(ROOT)) if path.exists() else "MISSING",
    })

artifact_table = pd.DataFrame(artifact_status)

display(artifact_table)

# Also verify the main exported files
exported_files = [
    ROOT / "outputs" / "refresh_queue.csv",
    ROOT / "outputs" / "model_results.json",
    ROOT / "outputs" / "summary.json",
    ROOT / "outputs" / "model_report.md",
]

print("\nMain exported files:")
for path in exported_files:
    status = "OK" if path.exists() else "MISSING"
    print(f"{status}: {path.relative_to(ROOT)}")



Running 04_evaluate_and_export.py

Paper artifacts
---------------


,artifact,exists,path
0,action_mix.svg,True,outputs/charts/action_mix.svg
1,confidence_mix.svg,True,outputs/charts/confidence_mix.svg
2,top_reason_codes.svg,True,outputs/charts/top_reason_codes.svg
3,top_feature_importance.svg,True,outputs/charts/top_feature_importance.svg
4,trend_distribution.svg,True,outputs/charts/trend_distribution.svg



Main exported files:
OK: outputs/refresh_queue.csv
OK: outputs/model_results.json
OK: outputs/summary.json
OK: outputs/model_report.md


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [X] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [X] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
